In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# load data 
RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"
df_erp = pd.read_parquet(os.path.join(RAW_DIR, "erp_transactions.parquet"))
df_support = pd.read_parquet(os.path.join(RAW_DIR, "supporting_documents.parquet"))

In [18]:
df_erp.columns

Index(['SALESDOCUMENT', 'SALESOFFICE', 'SALESGROUP', 'CUSTOMERPAYMENTTERMS',
       'SHIPPINGCONDITION', 'SALESDOCUMENTTYPE', 'SALESORGANIZATION',
       'DISTRIBUTIONCHANNEL', 'ORGANIZATIONDIVISION', 'BILLINGCOMPANYCODE',
       'TRANSACTIONCURRENCY', 'CREATIONDATE', 'CREATIONTIME',
       'HEADERINCOTERMSCLASSIFICATION', 'SALESDOCUMENTITEM', 'PLANT',
       'SHIPPINGPOINT', 'SALESDOCUMENTITEMCATEGORY', 'PRODUCT', 'SOLDTOPARTY',
       'SHIPTOPARTY', 'BILLTOPARTY', 'PAYERPARTY',
       'ITEMINCOTERMSCLASSIFICATION', 'ADDRESSID_SOLDTOPARTY',
       'ADDRESSID_BILLTOPARTY', 'ADDRESSID_PAYERPARTY',
       'ADDRESSID_SHIPTOPARTY'],
      dtype='object')

In [3]:
# Extract Recency and Frequency from the ERP headers
df_erp['CREATIONDATE'] = pd.to_datetime(df_erp['CREATIONDATE'])
max_dataset_date = df_erp['CREATIONDATE'].max()

rf_features = df_erp.groupby('SOLDTOPARTY').agg(
    total_orders=('SALESDOCUMENT', 'nunique'), # F: Frequency
    last_order_date=('CREATIONDATE', 'max')    # R: Recency base
).reset_index()

rf_features['days_since_last_order'] = (max_dataset_date - rf_features['last_order_date']).dt.days
rf_features.rename(columns={'SOLDTOPARTY': 'customer_id'}, inplace=True)
rf_features['customer_id'] = rf_features['customer_id'].astype(str)

In [4]:
# 3. Extract Monetary (M) value from the Supporting Documents (Purchase Orders)
print("Extracting Monetary values from Purchase Orders...")

# Filter only for purchase orders to avoid double-counting invoices
purchase_orders = df_support[df_support['document_type'] == 'PURCHASE_ORDER'].copy()

# Ensure financial columns are numeric
purchase_orders['total_amount'] = pd.to_numeric(purchase_orders['total_amount'], errors='coerce').fillna(0)
purchase_orders['quantity'] = pd.to_numeric(purchase_orders['quantity'], errors='coerce').fillna(0)

m_features = purchase_orders.groupby('customer_id').agg(
    total_spend=('total_amount', 'sum'),                # M: Monetary
    avg_order_value=('total_amount', 'mean'),
    total_items_purchased=('quantity', 'sum')
).reset_index()

m_features['customer_id'] = m_features['customer_id'].astype(str)

Extracting Monetary values from Purchase Orders...


In [9]:
# 4. Combine R, F, and M into one tabular dataset
tabular_features = pd.merge(rf_features, m_features, on='customer_id', how='left')

# Fill zeros for any customers that have ERP records but no matching PDF Purchase Orders yet
tabular_features = tabular_features.fillna({
    'total_spend': 0, 
    'avg_order_value': 0, 
    'total_items_purchased': 0
})

print(f"Generated tabular features for {len(tabular_features)} unique customers.")

Generated tabular features for 13155 unique customers.


In [10]:
# load nlp features
nlp_features = pd.read_parquet(os.path.join(PROCESSED_DIR, "nlp_customer_features.parquet"))
nlp_features['customer_id'] = nlp_features['customer_id'].astype(str)

# merge tabular data with NLP data
master_df = pd.merge(tabular_features, nlp_features, on='customer_id', how='inner')

In [11]:
master_df

,customer_id,total_orders,last_order_date,days_since_last_order,total_spend,avg_order_value,total_items_purchased,avg_sentiment,min_sentiment,total_urgent_emails,total_emails_sent
0,0001114321,3,2020-04-06,85,0.0,0.0,0.0,0.547109,0.4927,0,11
1,0002992449,1,2019-09-16,288,0.0,0.0,0.0,0.189541,0.0000,0,39
2,0003062521,1,2020-01-27,155,0.0,0.0,0.0,0.532600,0.4927,0,9
3,0003264409,76,2020-03-05,117,0.0,0.0,0.0,0.380400,0.0258,0,15
4,0004078317,1,2019-07-15,351,0.0,0.0,0.0,0.182837,0.0000,0,19
...,...,...,...,...,...,...,...,...,...,...,...
13150,9996676077,1,2018-07-15,716,0.0,0.0,0.0,0.051200,0.0000,0,5
13151,9996894830,28,2020-06-02,28,0.0,0.0,0.0,0.376350,0.0258,0,6
13152,9997075715,1,2018-04-18,804,0.0,0.0,0.0,0.064000,0.0000,0,4
13153,9999163225,89,2020-06-26,4,0.0,0.0,0.0,0.551100,0.4019,0,6


In [15]:
# churn threshold - 60 days
CHURN_THRESHOLD_DAYS = 60
master_df['is_churned'] = (master_df['days_since_last_order'] > CHURN_THRESHOLD_DAYS).astype(int)

# print class balance
churn_dist = master_df['is_churned'].value_counts(normalize=True) * 100
print(f"Active Customers (0): {churn_dist.get(0, 0.0):.1f}%")
print(f"Churned Customers (1): {churn_dist.get(1, 0.0):.1f}%")

# save final training set
output_path = os.path.join(PROCESSED_DIR, "master_training_set.parquet")
master_df.drop(columns=['last_order_date']).to_parquet(output_path, index=False)

Active Customers (0): 31.1%
Churned Customers (1): 68.9%

Ready for model training. Saved to: ../data/processed/master_training_set.parquet
